# Phase 4 — Google Vision API OCR Accuracy

**Purpose:** Test Google Vision OCR on 20 real product photos, visualize results side-by-side,
and document failure cases.

**Prerequisites:**
- Docker API running (default http://localhost:8000)
- 20 product JPEGs in `backend/test_photos/`
- Google service account at `backend/credentials/shelf-love-353bbeca17ab.json`

In [ ]:
import os, sys, json, time
from pathlib import Path
from io import BytesIO

import requests
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

In [ ]:
# === CONFIGURATION ===
API_BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:8000")

# Test user — the notebook will register this user if they don’t exist
TEST_EMAIL = "ocr_test@example.com"
TEST_PASSWORD = "test123"
TEST_NAME = "OCR Test"

# Image directory (this notebook is at backend/notebooks/)
NOTEBOOK_DIR = Path.cwd()
BACKEND_DIR = NOTEBOOK_DIR.parent
IMAGE_DIR = BACKEND_DIR / "test_photos"
# Set credentials path for direct vision-service use (optional)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(
    BACKEND_DIR / "credentials" / "shelf-love-353bbeca17ab.json"
)

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
image_paths = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS])

print(f"API: {API_BASE_URL}")
print(f"Images dir: {IMAGE_DIR}")
print(f"Images found: {len(image_paths)}")
print(f"Credentials exist: {Path(os.environ['GOOGLE_APPLICATION_CREDENTIALS']).exists()}")

In [ ]:
# === LOGIN / REGISTER ===
def login_or_register(email, password, name):
    r = requests.post(f"{API_BASE_URL}/auth/login", json={"email": email, "password": password})
    if r.status_code == 200:
        print(f"Logged in as {email}")
        return r.json()["access_token"]
    print(f"Login failed ({r.status_code}), registering...")
    r = requests.post(f"{API_BASE_URL}/auth/register", json={"email": email, "password": password, "name": name})
    if r.status_code in (200, 201):
        print(f"Registered as {email}")
        return r.json()["access_token"]
    raise RuntimeError(f"Auth failed: {r.status_code} {r.text}")

TOKEN = login_or_register(TEST_EMAIL, TEST_PASSWORD, TEST_NAME)
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print("Token acquired")

In [ ]:
# === HELPER: UPLOAD IMAGE + RUN OCR ===
def upload_and_scan(image_path, verbose=True):
    filename = image_path.name
    # 1) Get presigned URL
    r = requests.post(f"{API_BASE_URL}/uploads/presigned-url",
        json={"file_name": filename, "content_type": "image/jpeg"}, headers=HEADERS)
    r.raise_for_status()
    ud = r.json()
    if verbose:
        print(f"  presigned URL OK")
    # 2) Upload to S3
    with open(image_path, "rb") as f:
        data = f.read()
    r = requests.put(ud["upload_url"], data=data, headers={"Content-Type": "image/jpeg"})
    if r.status_code not in (200, 201, 204):
        raise RuntimeError(f"S3 upload failed: {r.status_code}")
    if verbose:
        print(f"  uploaded to S3: {ud['file_key']}")
    # 3) Process (OCR)
    r = requests.post(f"{API_BASE_URL}/uploads/process",
        json={"file_key": ud["file_key"], "barcode": None}, headers=HEADERS)
    r.raise_for_status()
    pd = r.json()
    if verbose:
        print(f"  OCR done: {len(pd.get('raw_ocr_text') or '')} chars, scan_id={pd.get('scan_id')}")
    return {
        "filename": filename,
        "file_key": ud["file_key"],
        "public_url": ud["public_url"],
        "scan_id": pd.get("scan_id"),
        "raw_ocr_text": pd.get("raw_ocr_text") or "",
        "success": True,
    }

In [ ]:
# === RUN OCR ON ALL IMAGES ===
results = []
print(f"Processing {len(image_paths)} images...\n")
for i, img_path in enumerate(image_paths, 1):
    print(f"[{i}/{len(image_paths)}] {img_path.name}")
    try:
        res = upload_and_scan(img_path)
        results.append(res)
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({
            "filename": img_path.name, "file_key": None, "public_url": None,
            "scan_id": None, "raw_ocr_text": "", "success": False,
        })
    print()

ok = sum(1 for r in results if r["success"])
print(f"\nDone. {ok}/{len(results)} succeeded.")

In [ ]:
# === SUMMARY TABLE ===
df = pd.DataFrame(results)
df["chars"] = df["raw_ocr_text"].apply(len)
df["preview"] = df["raw_ocr_text"].apply(
    lambda t: t[:150].replace("\n", " ") + ("..." if len(t) > 150 else "")
)
display(df[["filename", "success", "scan_id", "chars", "preview"]])

In [ ]:
# === VISUAL GRID ===
# Shows each product photo with OCR text beneath it
n = len(results)
cols = 2
rows = (n + 1) // 2

fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 6))
axes = axes.flatten() if rows * cols > 1 else [axes]

for idx in range(n):
    ax = axes[idx]
    res = results[idx]
    # Load image from S3
    if res["success"] and res["public_url"]:
        try:
            r = requests.get(res["public_url"], timeout=10)
            img = Image.open(BytesIO(r.content))
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f"Image error: {e}", ha="center", va="center")
    else:
        ax.text(0.5, 0.5, "FAILED", ha="center", va="center", fontsize=14, color="red")
    ax.set_title(f"{res['filename']}  (scan {res['scan_id']})", fontsize=9, fontweight="bold")
    ax.axis("off")
    # OCR text below
    ocr_txt = res["raw_ocr_text"][:400] if res["raw_ocr_text"] else "(no text)"
    ax.text(0.5, -0.10, f"OCR: {ocr_txt}", ha="center", va="top",
            transform=ax.transAxes, fontsize=6.5, fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#ffffcc", edgecolor="#cccccc"))

for idx in range(n, len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# === CHECK THE DATABASE ===
ids = [str(r["scan_id"]) for r in results if r["scan_id"] is not None]
if ids:
    id_list = ", ".join(ids)
    print("Run this to inspect stored records:\n")
    print(f"docker compose -f docker-compose.dev.yml exec db psql "
          f"-U cosmetics_user -d cosmetics "
          f"-c \"SELECT id, image_s3_key, LEFT(raw_ocr_text, 200) "
          f"FROM scan_results WHERE id IN ({id_list}) ORDER BY id;\"")
else:
    print("No scan IDs to inspect.")

---
## Failure Case Documentation

After reviewing the grid above, fill in the table below for every image where
OCR produced poor or missing results.

| # | Filename | Failure Type | Observed Issue | Would Phase 5 fail? |
|---|----------|-------------|----------------|---------------------|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

**Failure type options:** `reflection`, `curved_surface`, `small_font`,
`low_contrast`, `angled`, `multi_language`, `blurry`, `other`

### Summary

**Overall accuracy:** (how many images produced usable text?)

**Most common failure:** (which failure type appeared most often?)

**Key insight for Phase 5:** (e.g. \"regex-only extraction will miss dates on X% of images;\nrecommend OpenCV contour detection for PAO symbol + LLM fallback\")

---
*Save this notebook after filling in failure documentation.*